<details>
<summary style="
    cursor:pointer;background:#f7f7fb;border: 1px solid #e5e7eb;
    padding:10px 12px;border-radius:10px;font-weight:900;">
📊 Diagram Guide
</summary>

---

Here’s how I’d diagram **your** repo (based on what’s inside `01_ops_command_center/sql/`: `stg/`, `int/`, `mart/`, plus `_qa/`, `validation/`, `mart/recon/`, `mart/controls/`). The goal is twofold:

1. make the project instantly legible to a hiring manager
2. make the modeling *self-correcting* for you (diagrams that force clean grains + keys)

## The diagram set you want (4 diagrams, max)

### Diagram A — “One-page pipeline” (the README hero)

**Purpose:** explain the whole system in 10 seconds.

Show:

* **Raw** (your source drops / raw tables)
* **stg** typed+flagged views
* **int** conformed/dedup “truth” tables
* **mart** dims + facts + KPIs
* **controls/recon/QA** as guardrails

This should be a simple left-to-right flow with 5 boxes. No table-level detail.

---

### Diagram B — Domain lineage (Sales / Ops / HR / Finance)

**Purpose:** show what models exist and how they connect (the “DAG view”).

Make **one per domain**, each with 8–15 nodes max. For you, the natural grouping is already in your folders:

* **Sales**

  * `stg/stg_pos_transactions` → `int/sales/int_pos_dedup` → `int/sales/int_pos_daily` → `mart/sales/fact_sales_pos_daily`
  * `stg/stg_sales_distributor` → `int/sales/int_sales_distributor_dedup` → `mart/sales/fact_sales_distributor_daily`
  * then your rollups: `mart/sales/fact_sales_daily`, `agg_sales_store_daily`, `fact_distribution_coverage`

* **Ops**

  * `stg/stg_inventory_erp` → `int/ops/int_inventory_snapshot_dedup` → `int/ops/int_inventory_conformed` → `mart/ops/fact_inventory_snapshot_daily` → `kpi_instock_rate_daily`, `kpi_days_of_supply`
  * `stg/stg_sku_distribution_status` → `int/core/int_sku_distribution_status_dedup` → `mart/ops/fact_sku_distribution_status_daily`
  * `stg/stg_wms_shipments` → `mart/ops/fact_shipments_daily` (direct stg→mart is fine; just show it explicitly)

* **HR**

  * `stg/stg_timeclock_punches` → `int/hr/int_timeclock_punches_latest` → `mart/hr/fact_timeclock_punches`
  * `stg/stg_labor_payroll` → `int/hr/int_labor_*` → `mart/hr/fact_labor_*`

* **Finance**

  * `stg/stg_finance_actuals` → `mart/finance/fact_actuals_monthly`
  * `mart/finance/kpi_gross_margin_daily` (from `mart.fact_sales_distributor_daily`)
  * `mart/finance/recon_sales_to_gl_monthly`, `roi_monthly`, `breakeven_monthly`

---

### Diagram C — Star schema map (mart only)

**Purpose:** prove you understand dimensional modeling.

Keep this **only** to `mart.dim_*` and `mart.fact_*` and show the joins:

* `mart.dim_store`, `mart.dim_sku`, `mart.dim_date`, `mart.dim_employee`
* facts: sales / inventory / shipments / labor / actuals
* KPIs can be shown as derived from facts (dashed line), or omitted

---

### Diagram D — Trust & QA (controls + recon + validation)

**Purpose:** show you ship *reliable* analytics, not just queries.

Show:

* `_qa/_run_qa.sql` runs
* `mart/controls/*` checks (freshness, rowcounts, dim join coverage)
* `mart/recon/*` reconciliations
* `validation/*` semantic + reconciliation queries

This diagram is small but it screams “I’m employable.”

---

## What tools to use (so it stays maintainable)

* **Mermaid in Markdown** for Diagrams A + B (fast, versioned, renders on GitHub)
* **draw.io (diagrams.net)** or **Figma** for Diagram C (star schema is easier to layout visually)
* Optional: if you later move this into dbt, **dbt docs** becomes your auto-lineage graph

💡💡 Rule: Mermaid for *lineage*, draw.io/Figma for *relationships*.

---

## A concrete Mermaid template you can paste today

### A) README pipeline (Diagram A)

```mermaid
flowchart LR
  RAW[(raw tables / source drops)]
  STG[stg: typed + normalized + QA flags]
  INT[int: dedup + conformed truth tables]
  MART[mart: dims + facts + KPIs]
  TRUST[controls + recon + QA + validation]

  RAW --> STG --> INT --> MART --> TRUST
```

### B) Sales lineage (Diagram B — Sales)

```mermaid
flowchart LR
  subgraph STG[stg]
    stg_pos[stg.stg_pos_transactions]
    stg_dist[stg.stg_sales_distributor]
  end

  subgraph INT[int]
    int_pos_dedup[int.int_pos_dedup]
    int_pos_daily[int.int_pos_daily]
    int_dist_dedup[int.int_sales_distributor_dedup]
    int_sales_conf[int.int_sales_conformed]
  end

  subgraph MART[mart]
    dim_store[mart.dim_store]
    dim_sku[mart.dim_sku]
    fact_pos[mart.fact_sales_pos_daily]
    fact_dist[mart.fact_sales_distributor_daily]
    fact_sales[mart.fact_sales_daily]
    agg_store[mart.agg_sales_store_daily]
    cov[mart.fact_distribution_coverage]
  end

  stg_pos --> int_pos_dedup --> int_pos_daily --> fact_pos
  stg_dist --> int_dist_dedup --> fact_dist
  fact_pos --> int_sales_conf --> fact_sales --> agg_store --> cov

  dim_store -. joins .- fact_sales
  dim_sku -. joins .- fact_sales
```

(You can repeat this pattern for Ops/HR/Finance.)

---

## The “model correctly” guide (the part that prevents spaghetti)

### 1) Start from mart, then work backward

Pick the marts you’re promising:

* dims: `dim_store`, `dim_sku`, `dim_date`, `dim_employee`
* facts: sales / inventory / shipments / labor / actuals
* KPIs: instock, days of supply, gross margin, sales per labor hour

Then ask: **what does each mart object need upstream to be true?**
That answer defines `int`, and `int` defines `stg`.

### 2) Enforce grains like a paranoid scientist

Every model needs a declared grain and a key that actually matches it.

A practical rule-set:

* **stg**: same grain as raw (usually event-level), just typed/cleaned/flagged
* **int**: one “truth” per entity or per chosen time grain

  * examples you already have: “latest” (`int_dispensary_latest`, `int_timeclock_punches_latest`), “dedup” (`int_pos_dedup`), “daily” (`int_pos_daily`)
* **mart facts**: business grains (daily/monthly), with consistent date columns (`sale_date`, `as_of_date`, etc.)
* **mart dims**: one row per business key (store_code, sku, employee_id…)

💡💡 If a model’s grain isn’t obvious from the name, rename it.

### 3) Conform keys in **int**, not in mart

Your marts should “trust” that:

* `store_code` means the same thing everywhere
* `sku` means the same thing everywhere
* dates are normalized

That’s what `int` is for. Mart is where you **present**, not where you **negotiate reality**.

### 4) Make “direct stg → mart” a deliberate exception

You already do this (e.g., shipments). That’s fine when:

* it’s single-source
* the grain is stable and explicit
* you’re not joining across systems there

When it starts needing cross-source alignment, graduate it into `int`.

### 5) Treat QA/controls/recon as first-class models

Diagram D isn’t decoration. It’s your “trust layer.”

* `mart/controls/*` = “Are the tables sane today?”
* `mart/recon/*` = “Do numbers agree across sources?”
* `_qa/*` = “Do we fail fast?”

That’s how your project reads like a real analytics platform.

---

## Where to put the diagrams in your repo

* `README.md` → Diagram A (pipeline) + link to docs
* `docs/diagrams/lineage_sales.md` (Mermaid)
* `docs/diagrams/lineage_ops.md`
* `docs/diagrams/lineage_hr.md`
* `docs/diagrams/lineage_finance.md`
* `docs/diagrams/star_schema.drawio` (export PNG into same folder)
* `docs/diagrams/trust_layer.md` (Mermaid)

💡💡 This layout makes your repo feel “finished” because readers can navigate the system like a map.

If you follow this, your diagrams won’t just *describe* the project — they’ll actively force the project to stay well-modeled as you add more marts.

---

</details>

<details>
<summary style="
    cursor:pointer;background:#f7f7fb;border: 1px solid #e5e7eb;
    padding:10px 12px;border-radius:10px;font-weight:900;">
Should I model what columns I'm using from each in the lineage?
</summary>

<p>Yes, it's a good idea to model what columns you're using from each in the lineage. This will help you understand what data you're using and make it easier to maintain the lineage.</p>

Usually: **no, not in the main lineage diagram.** Column-level lineage turns a useful map into a bowl of spaghetti pretty fast.

What you *should* do is a **two-layer approach**:

**Layer 1 (always): model-level lineage**

* Boxes = models (stg/int/mart)
* Arrows = dependencies
* Annotate each box with:

  * **Grain** (1 row per what?)
  * **Primary key(s)**
  * **Time column** (`sale_date`, `as_of_date`, etc.)
    This is the diagram people will actually read.

**Layer 2 (selective): column-level mapping, but only for “high-stakes” fields**
Do column-level lineage only for columns that are:

* **business-critical** (revenue, units, on_hand_qty, cogs, margin)
* **keys** (store_code, sku, employee_id, txn_id)
* **definition-prone** (net_sales vs gross_sales, “orders”, “active store”, etc.)
* **conformed fields** (anything standardized in `int`, especially codes/names/statuses)

And instead of drawing a monster diagram, capture it as **a compact “mapping spec” per mart table**.

### The best format: a per-model mapping table (docs)

For each mart fact/dim, make a small doc like:

**`docs/mappings/mart.fact_sales_daily.md`**

* Grain
* Keys
* Measures (with formulas)
* Column provenance (source model + expression)

Example pattern:

* `sale_date` ← `int_sales_conformed.sale_date`
* `store_code` ← `int_sales_conformed.store_code`
* `sku` ← `int_sales_conformed.sku`
* `units_sold` ← `sum(int_sales_conformed.qty)`
* `net_sales` ← `sum(int_sales_conformed.net_amount)`
* `channel` ← `coalesce(int_sales_conformed.channel,'UNKNOWN')`

That gives you “column lineage” without diagram chaos.

### When column-level lineage *is* worth diagramming

Only in special cases:

* You’re explaining a tricky transformation (like dedup logic, or recon logic)
* You’re documenting a KPI with a contested definition (gross margin, instock rate)
* You have multiple sources merging and you need to show precedence rules

In those cases, do a **mini diagram** for just that metric/table.

💡💡 A clean rule: **diagram models, document columns.**
Your future self (and any hiring manager) will thank you for not inventing the world’s most annoying flowchart.

---

</details>

<details>
<summary style="
    cursor:pointer;background:#f7f7fb;border: 1px solid #e5e7eb;
    padding:10px 12px;border-radius:10px;font-weight:900;">
How do I model the diagram to show the relationships of tables from raw > stg > int > mart
</summary>

Model it like a **lineage DAG** (dependencies) with **layer swimlanes**, and only sprinkle in “relationships” (join keys) where they matter.

The trick: **relationships mean different things in different layers**:

* **raw → stg → int → mart:** “relationship” = **derivation / dependency** (this model is built from that model)
* **inside mart:** “relationship” = **join path** (fact ↔ dim via keys)

So your diagram should show *both*, but in a controlled way.

## Step-by-step: how to design the diagram

### 1) Pick the scope (don’t boil the ocean)

Do this in two levels:

* **Level 1 (README hero):** 5–15 nodes total, “how data flows”
* **Level 2 (domain diagrams):** Sales / Ops / HR / Finance each gets its own DAG

### 2) Use swimlanes for layers

Four columns (or four horizontal bands):

**RAW | STG | INT | MART**

That instantly communicates architecture without anyone reading a word.

### 3) Put *models* as nodes, *dependencies* as arrows

* One node per table/view/model.
* Arrow means: “built from” (SELECT/JOIN/AGG).

### 4) Annotate nodes with *grain + keys* (this is what makes it “modeled correctly”)

Under each node name, include:

* **Grain:** `1 row per day-store-sku`
* **Keys:** `sale_date, store_code, sku`
* Optional: “Type” label: `dedup`, `latest`, `daily`, `monthly`

### 5) Show join relationships only where it helps (mostly in mart)

Inside mart, show:

* **facts → dims** with dashed lines, labeled by join keys
  Example: `fact_sales_daily -- store_code --> dim_store`

That keeps your cross-layer diagram from becoming a hairball.

---

## A practical template (Mermaid) for raw → stg → int → mart

Paste this into a Markdown file (GitHub renders it):

```mermaid
flowchart LR
  %% ===== Swimlanes =====
  subgraph RAW[raw]
    raw_pos[(raw.pos_transactions)]
    raw_erp[(raw.erp_inventory_snapshot)]
  end

  subgraph STG[stg]
    stg_pos[stg.stg_pos_transactions\n(grain: txn line)\n(keys: txn_id, line_id)]
    stg_inv[stg.stg_inventory_snapshot\n(grain: snapshot row)\n(keys: as_of_date, store_code, sku)]
  end

  subgraph INT[int]
    int_pos_dedup[int.int_pos_dedup\n(grain: txn line)\n(keys: txn_id, line_id)]
    int_sales_daily[int.int_sales_daily\n(grain: day-store-sku)\n(keys: sale_date, store_code, sku)]
    int_inv_conf[int.int_inventory_conformed\n(grain: day-store-sku)\n(keys: as_of_date, store_code, sku)]
  end

  subgraph MART[mart]
    dim_store[mart.dim_store\n(grain: store)\n(keys: store_code)]
    dim_sku[mart.dim_sku\n(grain: sku)\n(keys: sku)]
    fact_sales[mart.fact_sales_daily\n(grain: day-store-sku)]
    fact_inv[mart.fact_inventory_snapshot_daily\n(grain: day-store-sku)]
    kpi_instock[mart.kpi_instock_rate_daily\n(grain: day-store)]
  end

  %% ===== Lineage (built-from) =====
  raw_pos --> stg_pos --> int_pos_dedup --> int_sales_daily --> fact_sales
  raw_erp --> stg_inv --> int_inv_conf --> fact_inv --> kpi_instock

  %% ===== Mart relationships (joins) =====
  fact_sales -. "store_code" .-> dim_store
  fact_sales -. "sku" .-> dim_sku
  fact_inv  -. "store_code" .-> dim_store
  fact_inv  -. "sku" .-> dim_sku
```

### How to adapt it to your repo fast

For each mart table you care about:

1. Put it in the **MART** lane
2. Add its **direct parents** in INT
3. Add upstream STG + RAW parents
4. Label the grain + keys
5. Add dashed joins to dims if it’s a fact

---

## A clean convention that prevents diagram chaos

* **Solid arrows** = “built from / lineage”
* **Dashed arrows** = “joins / relationships”
* **Node text** = `name + grain + keys`
* **One diagram per domain** once the node count goes above ~15

💡💡 If your diagram starts needing zoom to read, split it. A diagram that needs zoom is a diagram that’s secretly a punishment.

This approach makes it obvious you understand both:

* pipeline architecture (raw→stg→int→mart)
* dimensional modeling (facts/dims + conformed keys)

And it stays maintainable as you add models.

</details>